In [ ]:
# Import libraries
from IPython.display import display
import ipywidgets as ipw
from src import utils
import subprocess
import sys
import os

In [ ]:
CONFIG_FILENAME = "config/config.json"
CONFIG = utils.read_json(CONFIG_FILENAME)
OPENBIS_SESSION, SESSION_DATA = utils.connect_openbis_aiida()

openbis_connection_status_htmlbox = ipw.HTML(value="")
tab_children, tab_titles = [], []
if OPENBIS_SESSION:
    tabs_data = CONFIG["home_page"]["enable_links"]
    openbis_connection_status_htmlbox.value = CONFIG["home_page"]["enable_status"]
else:
    tabs_data = CONFIG["home_page"]["disable_links"]
    openbis_connection_status_htmlbox.value = CONFIG["home_page"]["disable_status"]

# Populate tab widget
for tab_title, html_content in tabs_data.items():
    tab_titles.append(tab_title)
    content = "".join(html_content)
    widgets_list = [ipw.HTML(value=content)]
    vbox = ipw.VBox(widgets_list)
    tab_children.append(vbox)

tabs = ipw.Tab(children=tab_children)
for idx, title in enumerate(tab_titles):
    tabs.set_title(idx, title)

In [ ]:
# This code creates a process in the backend to upload logs to AFS every minute.
# Define the path to your existing script
script_path = "src/upload_logs_to_afs.py"
pid_file = "logs/upload_logs.pid"

# Check if the process is already running to prevent duplicates
already_running = False
if os.path.exists(pid_file):
    with open(pid_file, "r") as f:
        pid = int(f.read().strip())
    try:
        os.kill(pid, 0)
        already_running = True
    except OSError:
        already_running = False

# Launch it completely detached from Jupyter
if not already_running:
    process = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )

    # Save the Process ID (PID) so we can stop it later if needed
    with open(pid_file, "w") as f:
        f.write(str(process.pid))

# AiiDAlab-openBIS app

In [ ]:
display(openbis_connection_status_htmlbox)

In [ ]:
display(tabs)